# FarmTech Solutions — Visão Computacional com YOLOv5
## Detecção de Milho e Tomate

Este notebook implementa um sistema de visão computacional utilizando YOLOv5
para detectar e classificar imagens de **milho** e **tomate**, simulando um
cenário real de uso na área do agronegócio.

As etapas seguidas foram:
1. Montagem e organização do dataset
2. Rotulação das imagens com Make Sense IA
3. Treinamento com 30 épocas
4. Treinamento com 60 épocas
5. Comparação dos resultados
6. Teste do modelo final

In [ ]:
# Configuração para rodar localmente com caminhos relativos
import os
from pathlib import Path


def encontrar_raiz_projeto() -> Path:
    """Encontra a raiz que contem data/data.yaml, rodando da raiz ou de notebook/."""
    cwd = Path.cwd().resolve()
    candidatos = [cwd, *cwd.parents]
    for candidato in candidatos:
        if (candidato / "data" / "data.yaml").exists() and (candidato / "notebook").exists():
            return candidato
    raise FileNotFoundError(
        "Nao foi possivel encontrar a raiz do projeto. "
        "Abra o notebook a partir da raiz do projeto ou da pasta notebook/."
    )


PROJECT_ROOT = encontrar_raiz_projeto()
os.chdir(PROJECT_ROOT)

# Mantem caches/configuracoes dentro do projeto para evitar caminhos fixos da maquina local.
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib"))
os.environ.setdefault("JUPYTER_CONFIG_DIR", str(PROJECT_ROOT / ".jupyter"))
os.environ.setdefault("JUPYTER_DATA_DIR", str(PROJECT_ROOT / ".jupyter_data"))
os.environ.setdefault("JUPYTER_RUNTIME_DIR", str(PROJECT_ROOT / ".jupyter_runtime"))
os.environ.setdefault("IPYTHONDIR", str(PROJECT_ROOT / ".ipython"))

for local_dir in [".matplotlib", ".jupyter", ".jupyter_data", ".jupyter_runtime", ".ipython"]:
    Path(local_dir).mkdir(exist_ok=True)

for local_dir in [
    "results/yolo_custom/treino_30_epocas",
    "results/yolo_custom/treino_60_epocas",
    "results/yolo_custom/val_30_epocas",
    "results/yolo_custom/val_60_epocas",
    "results/yolo_custom/teste_30_conf15",
    "results/yolo_custom/teste_60_conf15",
    "results/yolo_padrao",
    "results/cnn_do_zero",
    "assets/prints_yolo_custom",
    "assets/prints_yolo_padrao",
    "assets/prints_cnn",
]:
    Path(local_dir).mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", PROJECT_ROOT)
print("data.yaml existe?", Path("data/data.yaml").exists())
print("data_yolov5.yaml existe?", Path("data/data_yolov5.yaml").exists())

## 1. Instalação do YOLOv5
Usamos o repositório oficial do YOLOv5 e instalamos as dependências necessárias.

In [ ]:
# Clonar o repositorio oficial do YOLOv5, se ainda nao existir
import sys
from pathlib import Path

if not Path("yolov5").exists():
    !git clone https://github.com/ultralytics/yolov5

# Instalar as dependencias necessarias no mesmo Python usado pelo notebook.
!{sys.executable} -m pip install -r yolov5/requirements.txt

## 2. Configuração do Dataset
Verificamos se o dataset está acessível pela estrutura local do projeto.

In [ ]:
from pathlib import Path

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
base_path = Path("data")

for split in ["train", "valid", "test"]:
    images_dir = base_path / split / "images"
    labels_dir = base_path / split / "labels"
    images = sorted(p for p in images_dir.glob("*") if p.suffix.lower() in image_exts)
    missing_labels = [p for p in images if not (labels_dir / f"{p.stem}.txt").exists()]

    print(f"\nSplit: {split}")
    print("Pasta de imagens existe?", images_dir.exists(), "-", images_dir)
    print("Pasta de labels existe?", labels_dir.exists(), "-", labels_dir)
    print("Quantidade de imagens:", len(images))
    print("Labels faltantes:", len(missing_labels))

    for path in missing_labels[:10]:
        print(" -", path.name)


## 3. Treinamento com 30 Épocas
Realizamos o primeiro treinamento utilizando 30 épocas para servir como baseline de comparação.

In [ ]:
# Desabilitar o wandb completamente
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

# Treinamento 1 — 30 épocas
!python yolov5/train.py --img 640 \
                 --batch 16 \
                 --epochs 30 \
                 --data data/data_yolov5.yaml \
                 --weights yolov5s.pt \
                 --project results/yolo_custom \
                 --name treino_30_epocas \
                 --plots


## 4. Treinamento com 60 Épocas
Segundo treinamento com o dobro de épocas para comparar o desempenho com o modelo anterior.

In [ ]:
# Desabilitar o wandb completamente
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

# Treinamento 2 — 60 épocas
!python yolov5/train.py --img 640 \
                 --batch 16 \
                 --epochs 60 \
                 --data data/data_yolov5.yaml \
                 --weights yolov5s.pt \
                 --project results/yolo_custom \
                 --name treino_60_epocas \
                 --plots


## 5. Validação dos Modelos
Avaliamos os dois modelos treinados nas imagens de validação para comparar desempenho.

In [ ]:
# Validação do modelo treinado com 30 épocas
!python yolov5/val.py --img 640 \
               --data data/data_yolov5.yaml \
               --weights results/yolo_custom/treino_30_epocas/weights/best.pt \
               --project results/yolo_custom \
               --name val_30_epocas


In [ ]:
# Validação do modelo treinado com 60 épocas
!python yolov5/val.py --img 640 \
               --data data/data_yolov5.yaml \
               --weights results/yolo_custom/treino_60_epocas/weights/best.pt \
               --project results/yolo_custom \
               --name val_60_epocas


## 6. Teste do Modelo
Aplicamos os modelos de 30 e 60 epocas nas imagens de teste com `conf=0.15` e `save-txt` para visualizar as deteccoes e comparar falsos positivos.

In [ ]:
# Teste com o modelo de 30 épocas nas imagens de teste
!python yolov5/detect.py --img 640 \
                  --weights results/yolo_custom/treino_30_epocas/weights/best.pt \
                  --source data/test/images \
                  --conf 0.15 \
                  --save-txt \
                  --project results/yolo_custom \
                  --name teste_30_conf15


In [ ]:
# Teste com o modelo de 60 épocas nas imagens de teste
!python yolov5/detect.py --img 640 \
                  --weights results/yolo_custom/treino_60_epocas/weights/best.pt \
                  --source data/test/images \
                  --conf 0.15 \
                  --save-txt \
                  --project results/yolo_custom \
                  --name teste_60_conf15


## 7. Visualização dos Resultados
Exibimos as imagens de teste processadas pelo modelo para análise visual das detecções.

In [ ]:
import glob
from IPython.display import Image, display

print("=== Resultados - 30 épocas (conf=0.15) ===")
imgs_30 = glob.glob("results/yolo_custom/teste_30_conf15/*.jpg")
for img in sorted(imgs_30):
    display(Image(filename=img, width=500))

print("=== Resultados - 60 épocas (conf=0.15) ===")
imgs_60 = glob.glob("results/yolo_custom/teste_60_conf15/*.jpg")
for img in sorted(imgs_60):
    display(Image(filename=img, width=500))


In [ ]:
import shutil
from pathlib import Path

assets_path = Path("assets/prints_yolo_custom")
assets_path.mkdir(parents=True, exist_ok=True)

for img in sorted(Path("results/yolo_custom/teste_30_conf15").glob("*.jpg")):
    shutil.copy(img, assets_path / img.name)

print("Imagens salvas em:", assets_path)


## 8. Comparação e Conclusões

### Resultados da Validação
| Modelo | mAP50 | Precisão | Recall |
|--------|-------|----------|--------|
| 30 épocas | 0.465 | 0.288 | 0.625 |
| 60 épocas | 0.354 | 0.355 | 0.479 |

### Resultados dos Testes (conf=0.15)
| Configuração | Detecções corretas | Falsos positivos | Observação |
|--------------|-------------------|------------------|------------|
| 30 épocas | 5/8 | Nenhum | 1 milho + 4 tomates detectados corretamente |
| 60 épocas | 6/8 | Sim | Confundiu milho em imagens de tomate |

### Conclusões
- O modelo de **30 épocas** apresentou maior mAP50 na validação (0.465 vs 0.354)
- O modelo de **30 épocas** foi mais confiável nos testes, sem falsos positivos
- O modelo de **60 épocas** detectou mais imagens, porém com falsos positivos — identificou milho em imagens de tomate
- O **tomate** foi melhor aprendido pelo modelo do que o milho em ambos os treinamentos
- Reduzir o limiar de confiança de 0.25 para 0.15 aumentou as detecções sem comprometer demais a precisão

### Pontos Fortes
- Sistema capaz de detectar tomates com alta consistência (4/4 com conf=0.15)
- Treinamento rápido — menos de 1 hora para os dois modelos completos
- O modelo demonstra potencial real para aplicação no agronegócio

### Limitações
- Dataset reduzido (40 imagens por classe) limita a capacidade de generalização
- Milho foi detectado em apenas 1 de 4 imagens de teste com o modelo de 30 épocas
- Modelo de 60 épocas apresentou falsos positivos com conf=0.15
- Imagens de milho em plantação/campo foram mais difíceis de detectar do que espigas isoladas

## 9. Estrutura dos Arquivos de Resultado

Os resultados dos treinamentos foram salvos em:
- `results/yolo_custom/treino_30_epocas/` — pesos e métricas do modelo de 30 épocas
- `results/yolo_custom/treino_60_epocas/` — pesos e métricas do modelo de 60 épocas
- `results/yolo_custom/val_30_epocas/` — validação do modelo de 30 épocas
- `results/yolo_custom/val_60_epocas/` — validação do modelo de 60 épocas
- `results/yolo_custom/teste_30_conf15/` — teste do modelo de 30 épocas com conf=0.15
- `results/yolo_custom/teste_60_conf15/` — teste do modelo de 60 épocas com conf=0.15
- `assets/prints_yolo_custom/` — prints do YOLO customizado
- `results/yolo_padrao/` — inferências do YOLO padrão
- `results/cnn_do_zero/` — modelo, métricas e histórico da CNN


## 10. Entrega 2 — YOLO Padrão

Nesta etapa, utilizamos um modelo YOLO pré-treinado (sem customização) para avaliar o desempenho em comparação com o modelo treinado anteriormente.

In [ ]:
from ultralytics import YOLO
import shutil
import time
from pathlib import Path

TEST_DIR = Path("data/test/images")
RESULTS_DIR = Path("results/yolo_padrao")
PRINTS_DIR = Path("assets/prints_yolo_padrao")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PRINTS_DIR.mkdir(parents=True, exist_ok=True)

print("Existe?", TEST_DIR.exists())
print("Quantidade de imagens:", len(list(TEST_DIR.glob("*"))))

model = YOLO("yolov8n.pt")

inicio = time.time()

results = model.predict(
    source=str(TEST_DIR),
    save=True,
    project="results",
    name="yolo_padrao",
    exist_ok=True,
    conf=0.25
)

fim = time.time()
inference_time = fim - inicio
print(f"Tempo de inferência YOLO padrão: {inference_time:.2f} segundos")

# Algumas versões do Ultralytics salvam em runs/detect/... mesmo com project/name.
# Por isso copiamos o diretório real da predição para a pasta esperada da entrega.
actual_save_dir = Path(results[0].save_dir) if results else RESULTS_DIR
print("Diretório original gerado pelo YOLO:", actual_save_dir)

for img in sorted(actual_save_dir.glob("*.jpg")):
    shutil.copy2(img, RESULTS_DIR / img.name)
    shutil.copy2(img, PRINTS_DIR / img.name)

(RESULTS_DIR / "tempo_inferencia.txt").write_text(
    f"Tempo de inferência YOLO padrão: {inference_time:.2f} segundos\n",
    encoding="utf-8"
)

print("Resultados salvos em:", RESULTS_DIR)
print("Prints salvos em:", PRINTS_DIR)


In [ ]:
import glob
from IPython.display import Image, display

imagens = glob.glob("results/yolo_padrao/*.jpg")

for img in sorted(imagens)[:5]:
    display(Image(filename=img, width=500))


## 11. CNN Treinada do Zero

Nesta etapa, implementamos uma rede neural convolucional (CNN) do zero com TensorFlow/Keras para classificar as duas classes do dataset. A CNN usa a estrutura `data_cnn/`, salva o modelo, registra metricas em JSON e gera graficos de acuracia e loss.

In [ ]:
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

# Prepara data_cnn/train|valid|test com subpastas milho e tomate a partir do dataset YOLO.
# A pasta data_cnn e gerada automaticamente a partir de data/, sem depender de scripts externos.
from pathlib import Path
import shutil

SPLITS = ("train", "valid", "test")
CLASSES = ("milho", "tomate")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_class(image_name):
    normalized_name = image_name.lower()
    for class_name in CLASSES:
        if class_name in normalized_name:
            return class_name
    return None

source_root = Path("data")
target_root = Path("data_cnn")
total_copied = 0

for split in SPLITS:
    source_images = source_root / split / "images"

    for class_name in CLASSES:
        (target_root / split / class_name).mkdir(parents=True, exist_ok=True)

    for image_path in sorted(source_images.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        class_name = image_class(image_path.name)
        if class_name is None:
            continue

        shutil.copy2(image_path, target_root / split / class_name / image_path.name)
        total_copied += 1

print(f"data_cnn gerado com {total_copied} imagens")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 10
RESULTS_DIR = Path("results/cnn_do_zero")
PRINTS_DIR = Path("assets/prints_cnn")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PRINTS_DIR.mkdir(parents=True, exist_ok=True)

train_ds = tf.keras.utils.image_dataset_from_directory(
    "data_cnn/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=42
)

valid_ds = tf.keras.utils.image_dataset_from_directory(
    "data_cnn/valid",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "data_cnn/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
valid_ds = valid_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(2, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

inicio_treino = time.time()
history = model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS)
tempo_treino = time.time() - inicio_treino

test_loss, test_acc = model.evaluate(test_ds)

inicio_inferencia = time.time()
predictions = model.predict(test_ds)
tempo_inferencia = time.time() - inicio_inferencia

model.save(RESULTS_DIR / "cnn_do_zero.keras")

metrics = {
    "classes": class_names,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "tempo_treino_segundos": float(tempo_treino),
    "tempo_inferencia_segundos": float(tempo_inferencia),
    "quantidade_predicoes_teste": int(len(predictions)),
}
(RESULTS_DIR / "metricas.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")

plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="treino")
plt.plot(history.history["val_accuracy"], label="validacao")
plt.title("Acurácia da CNN")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "grafico_acuracia.png", dpi=150)
shutil.copy2(RESULTS_DIR / "grafico_acuracia.png", PRINTS_DIR / "grafico_acuracia.png")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="treino")
plt.plot(history.history["val_loss"], label="validacao")
plt.title("Loss da CNN")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "grafico_loss.png", dpi=150)
shutil.copy2(RESULTS_DIR / "grafico_loss.png", PRINTS_DIR / "grafico_loss.png")
plt.show()

print(f"Acurácia CNN: {test_acc:.2f}")
print(f"Loss CNN: {test_loss:.4f}")
print(f"Tempo de treinamento CNN: {tempo_treino:.2f} segundos")
print(f"Tempo de inferência CNN: {tempo_inferencia:.2f} segundos")
print("Modelo salvo em:", RESULTS_DIR / "cnn_do_zero.keras")
print("Métricas salvas em:", RESULTS_DIR / "metricas.json")
print("Gráficos salvos em:", RESULTS_DIR)


## 12. Comparação entre os Modelos

| Critério | YOLO Customizado | YOLO Padrão | CNN do Zero |
|---|---|---|---|
| Facilidade de uso/integração | Média: exige dataset rotulado em formato YOLO, configuração do treinamento e ajuste dos pesos | Alta: utiliza modelo pré-treinado (`yolov8n.pt`) e requer apenas chamada de inferência | Média: exige reorganização do dataset em pastas por classe e implementação/treinamento da rede |
| Precisão esperada | Maior para o problema específico, pois foi treinado com imagens de milho e tomate | Limitada para este cenário, pois o modelo foi treinado em classes genéricas do COCO | Limitada pelo baixo volume de dados e por realizar classificação global da imagem |
| Tempo de treinamento/customização | Maior: requer treinamento customizado com 30/60 épocas e labels YOLO | Não requer treinamento adicional; apenas download/carregamento do modelo pré-treinado | Menor que o YOLO customizado, mas ainda exige treinamento supervisionado do zero |
| Tempo de inferência | Rápido após o treinamento, retornando classe, confiança e bounding box | Rápido, pois usa pesos pré-treinados e não passa por etapa de customização | Rápido em poucas imagens, mas retorna somente a classe da imagem |
| Inferência | Detecta objetos específicos do projeto, retornando classe, confiança e coordenadas da bounding box | Detecta objetos baseados no dataset COCO, podendo não reconhecer corretamente classes específicas como milho e tomate | Classifica a imagem inteira como milho ou tomate, sem localizar o objeto |
| Pontos fortes | Melhor adequação ao problema; detecta e localiza as classes de interesse | Fácil integração, execução rápida e útil como baseline comparativo | Arquitetura simples, didática e adequada para demonstrar classificação supervisionada |
| Limitações | Depende da qualidade das labels e de maior volume de dados para generalizar melhor | Não foi treinado especificamente para milho e tomate, podendo gerar classes incorretas | Não gera bounding boxes, sofre com poucos dados e tem menor capacidade de generalização |

### Conclusão Final
O **YOLO customizado** foi o modelo mais adequado para a proposta, porque foi treinado especificamente para **milho** e **tomate** e retorna bounding boxes com localização e confiança. Isso atende melhor ao objetivo de visão computacional do projeto, pois além de classificar, também localiza os objetos na imagem.

O **YOLO padrão** foi útil como baseline por ser simples de integrar e rápido para inferência, mas não foi treinado especificamente para as classes do projeto. Por isso, ele pode falhar ou retornar classes genéricas que não representam corretamente milho e tomate.

A **CNN treinada do zero** funcionou como abordagem de classificação, mas não localiza objetos e depende mais do volume de dados para generalizar bem. Ela é útil para comparação didática, mas é menos indicada quando a entrega precisa detectar a posição dos objetos na imagem.